le nom: El jattioui
le prenom:Maryame
Master:GLCC

In [1]:
import numpy as np

# =================================================================
# 1. DATASET D'ENTRAÎNEMENT (Input Data)
# =================================================================
# X_train représente la matrice des caractéristiques (n_samples, n_features)
# Colonne 0 : Moyenne du dossier académique (sur 20)
# Colonne 1 : Note de l'entretien oral de sélection (sur 20)
X_train = np.array([
    [15.0, 14.0], [10.5, 11.0], [18.0, 17.0], [8.0, 9.0],  [12.0, 13.0], 
    [7.0,  6.5],  [16.0, 15.5], [9.5,  10.0], [14.0, 14.5], [6.0,  5.0]
])

# y_train représente le vecteur cible des étiquettes (Labels)
# 1 = Étudiant Admis au Master | 0 = Étudiant Refusé
y_train = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

# =================================================================
# 2. L'ALGORITHME NAIVE BAYES GAUSSIEN (From Scratch)
# =================================================================
class MyGaussianNaiveBayes:
    """
    Classifieur Bayésien Naïf adapté aux variables continues.
    Il modélise la distribution de chaque caractéristique à l'aide d'une
    loi normale (Gaussienne) caractérisée par sa moyenne (mu) et sa variance (sigma^2).
    """
    def __init__(self):
        self.classes = None    # Liste des classes uniques présentes (ex: [0, 1])
        self.mean = {}         # Dictionnaire stockant les vecteurs de moyennes par classe
        self.var = {}          # Dictionnaire stockant les vecteurs de variances par classe
        self.priors = {}       # Dictionnaire stockant les probabilités à priori P(Y)

    def fit(self, X, y):
        """
        Phase d'apprentissage : Extraction des métriques statistiques globales.
        Complexité temporelle : O(n_samples * n_features)
        """
        n_samples, n_features = X.shape
        self.classes = np.unique(y) # Identification des classes uniques (0 et 1)

        # On calcule les paramètres de la distribution pour CHAQUE classe de manière isolée
        for c in self.classes:
            # ÉTAPE A : Filtrage des données appartenant uniquement à la classe courante 'c'
            X_c = X[y == c]

            # ÉTAPE B : Calcul de la probabilité à priori P(Y = c)
            # Formule : Fréquence d'apparition de la classe dans le dataset d'entraînement
            self.priors[c] = X_c.shape[0] / float(n_samples)

            # ÉTAPE C : Calcul des estimateurs du maximum de vraisemblance (Moyenne et Variance)
            # np.mean/np.var avec axis=0 calcule la statistique colonne par colonne (par caractéristique)
            self.mean[c] = np.mean(X_c, axis=0)
            self.var[c] = np.var(X_c, axis=0)

    def _calculate_gaussian_likelihood(self, class_idx, x):
        """
        Calcule la vraisemblance conditionnelle P(Xi | Y = c) pour un échantillon 'x'.
        Elle s'appuie sur la fonction de densité de la Loi Normale de Gauss.
        
        Formule mathématique :
        P(x_i | c) = [1 / sqrt(2 * pi * sigma^2)] * exp( - (x_i - mu)^2 / (2 * sigma^2) )
        """
        mean = self.mean[class_idx] # Vecteur des moyennes pour la classe 'c'
        var = self.var[class_idx]   # Vecteur des variances pour la classe 'c'
        
        # SÉCURITÉ INFORMATIQUE : Ajout d'un régularisateur "epsilon" (smoothing)
        # Évite la division par zéro si la variance d'une caractéristique est nulle.
        eps = 1e-6
        var = var + eps

        # Évaluation mathématique du numérateur (la fonction exponentielle)
        # Remarque : Grâce à NumPy, le calcul se fait en parallèle pour toutes les colonnes de 'x'
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        
        # Évaluation mathématique du dénominateur (le facteur de normalisation)
        denominator = np.sqrt(2 * np.pi * var)
        
        # Renvoie un vecteur contenant la vraisemblance P(X_i | c) de chaque variable
        return numerator / denominator

    def _predict_single_sample(self, x):
        """
        Prédit l'étiquette d'une seule ligne de données en appliquant la règle du
        Maximum à Posteriori (MAP).
        """
        posteriors = []

        for c in self.classes:
            # ÉTAPE A : Initialisation avec le logarithme de la probabilité à priori
            # Multiplier des probabilités (ex: 0.01 * 0.002 * 0.05) provoque un "Underflow" (arrondi à 0).
            # Grâce aux propriétés des logs : log(a * b) = log(a) + log(b). On remplace le produit par une somme.
            prior = np.log(self.priors[c])

            # ÉTAPE B : Calcul des vraisemblances P(X_i | Y = c)
            likelihoods = self._calculate_gaussian_likelihood(c, x)

            # ÉTAPE C : Application de l'hypothèse d'indépendance "Naïve"
            # On suppose que les colonnes n'ont aucun lien entre elles sachant la classe.
            # On prend le log de chaque vraisemblance, et on fait la somme géométrique (np.sum).
            # On ajoute 1e-9 à l'intérieur du log pour éviter l'erreur fatale log(0).
            conditional_sum = np.sum(np.log(likelihoods + 1e-9))

            # ÉTAPE D : Calcul de la valeur finale non normalisée de la probabilité à posteriori
            # Log( P(Y|X) ) est proportionnel à : Log( P(Y) ) + Somme( Log( P(X_i|Y) ) )
            posterior = prior + conditional_sum
            
            # Stockage du couple (score, classe)
            posteriors.append((posterior, c))

        # ÉTAPE E : Sélection de la classe maximisant la fonction objectif (Règle MAP)
        # La fonction lambda permet de trier et de renvoyer l'élément ayant le plus grand score 'posterior'
        return max(posteriors, key=lambda item: item[0])[1]

    def predict(self, X):
        """
        Point d'entrée pour la prédiction d'une matrice d'individus.
        Parcourt itérativement chaque échantillon pour lui attribuer une classe.
        """
        return np.array([self._predict_single_sample(x) for x in X])

# =================================================================
# 3. PIPELINE DE TEST ET LOGS DE VALIDATION
# =================================================================
if __name__ == "__main__":
    # Instanciation de notre classe personnalisée
    clf_nb = MyGaussianNaiveBayes()
    
    # Entraînement du modèle sur nos données de Master
    clf_nb.fit(X_train, y_train)

    print("=" * 60)
    print("MÉTRIQUES D'APPRENTISSAGE (Moyennes & Variances Apprises)")
    print("=" * 60)
    for c in clf_nb.classes:
        label_text = "ADMIS" if c == 1 else "REJETÉ"
        print(f"Classe [{label_text}] :")
        print(f"  -> Probabilité Prior P(Y) : {clf_nb.priors[c]:.2f}")
        print(f"  -> Vecteur des Moyennes [Dossier, Entretien] : {clf_nb.mean[c]}")
        print(f"  -> Vecteur des Variances [Dossier, Entretien] : {clf_nb.var[c]}\n")

    # Définition de nouveaux profils d'étudiants (Données inconnues du modèle)
    # Étudiant 1 : Dossier solide (14.5) et Entretien réussi (15.0) -> Devrait être admis
    # Étudiant 2 : Dossier faible (7.5) et Entretien moyen (8.0)    -> Devrait être refusé
    X_test = np.array([
        [14.5, 15.0],
        [7.5, 8.0]
    ])

    # Inférence (Prédiction)
    predictions = clf_nb.predict(X_test)

    print("=" * 60)
    print("RÉSULTATS DE L'INFÉRENCE (Prédictions sur Nouveaux Profils)")
    print("=" * 60)
    for i, sample in enumerate(X_test):
        decision = "ADMIS (Classe 1)" if predictions[i] == 1 else "REJETÉ (Classe 0)"
        print(f"Étudiant n°{i+1} {sample} | Statut Prédit : {decision}")

MÉTRIQUES D'APPRENTISSAGE (Moyennes & Variances Apprises)
Classe [REJETÉ] :
  -> Probabilité Prior P(Y) : 0.50
  -> Vecteur des Moyennes [Dossier, Entretien] : [8.2 8.3]
  -> Vecteur des Variances [Dossier, Entretien] : [2.66 4.96]

Classe [ADMIS] :
  -> Probabilité Prior P(Y) : 0.50
  -> Vecteur des Moyennes [Dossier, Entretien] : [15.  14.8]
  -> Vecteur des Variances [Dossier, Entretien] : [4.   1.86]

RÉSULTATS DE L'INFÉRENCE (Prédictions sur Nouveaux Profils)
Étudiant n°1 [14.5 15. ] | Statut Prédit : ADMIS (Classe 1)
Étudiant n°2 [7.5 8. ] | Statut Prédit : REJETÉ (Classe 0)
